# Discrete Manifold Navigation
This notebook provides a detailed, mathematically rigorous walkthrough of the **Holonomic Association Model (HAM)**.
The HAM library treats differentiable **Finsler Geometries** as learnable modules in JAX, allowing us to solve optimal control and simulation problems purely through geometric dynamics.

### Core Philosophy: Metric-First Design
In HAM, the `FinslerMetric` object is the single source of truth. We define the scalar **Energy Functional**:
$$\mathcal{E}[\gamma] = \int \frac{1}{2} F^2(x, \dot{x}) dt$$
where $F(x,v)$ is the Finsler cost function. Everything else—including the Geodesic Spray (the physics engine), curvature, and Berwald parallel transport—is auto-differentiated directly from $F$ using `jax.grad` and `jax.hessian`, entirely avoiding the $O(N^3)$ computational cost of expanding Christoffel symbols manually.

## 1. Discrete Manifolds in HAM
The Holonomic Association Model doesn't strictly require continuous, analytically-defined manifolds. It fully supports real-world topologies via the `TriangularMesh` class. 
This allows us to learn Finslerian costs and perform optimal navigation on discrete LiDAR point clouds or triangulated CAD models.


In [1]:
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
import numpy as np

from ham.geometry import TriangularMesh, Randers, Euclidean
from ham.solvers import AVBDSolver
from ham.vis import generate_icosphere


## 2. Generating a Discrete Environment
We generate a discrete mesh representing a sphere (an icosphere). Notice we are no longer using the `Sphere` continuous topology class.


In [2]:
print("Generating Discrete Mesh...")
verts, faces = generate_icosphere(radius=1.0, subdivisions=3)
mesh = TriangularMesh(verts, faces)


Generating Discrete Mesh...


## 3. Defining Discrete Wind
On a discrete mesh, the wind vector field $W^i(x)$ is defined globally as a function of the 3D position in the embedding space. The `DiscreteRanders` metric automatically projects these 3D vectors onto the local tangent planes of the individual faces to maintain topological validity.


In [3]:
def w_net(x): 
    base = jnp.array([-x[1], x[0], 0.0])
    return 0.8 * base

def h_net(x): return jnp.eye(3)

discrete_randers = Randers(mesh, h_net, w_net)
discrete_euclidean = Euclidean(mesh)


## 4. Discrete Trajectory Optimization
We define start and end points and rely on the same `AVBDSolver`. Because the metric uses the `TriangularMesh` backend, the Augmented Vertex Block Descent automatically transitions to calculating gradients over the discrete face topology, ensuring paths navigate edges safely without penetrating the surface!


In [4]:
start = jnp.array([1.0, 0.0, 0.0])
end   = jnp.array([0.0, 0.0, 1.0])

solver = AVBDSolver(step_size=0.05, beta=10.0, iterations=300)

print("Solving Discrete Riemannian (Shortest Path)...")
traj_eucl = solver.solve(discrete_euclidean, start, end, n_steps=40)

print("Solving Discrete Randers (Fastest Path with Wind)...")
traj_rand = solver.solve(discrete_randers, start, end, n_steps=40)


Solving Discrete Riemannian (Shortest Path)...
Solving Discrete Randers (Fastest Path with Wind)...


## 5. Interactive 3D Mesh Visualization
We use Plotly's `Mesh3d` to render the wireframe faceted surface. This interactive view allows you to verify that the discrete solver successfully mapped the geodesics directly across the mesh edges rather than penetrating the interior of the shape. Rotate the render to see the wind flow and the topology!


In [ ]:
fig = go.Figure()

# Plot the mesh faces (without alphahull, so it uses exact triangles)
fig.add_trace(go.Mesh3d(
    x=verts[:,0], y=verts[:,1], z=verts[:,2],
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    color='lightgray', opacity=0.3,
    lighting=dict(ambient=0.5, diffuse=0.8, roughness=0.5, specular=0.2),
    name='Mesh Faces'
))

# Extract and plot the actual triangular mesh wireframe edges
edges = set()
for f in faces:
    for idx in range(3):
        idx1 = int(f[idx])
        idx2 = int(f[(idx+1)%3])
        e = tuple(sorted((idx1, idx2)))
        edges.add(e)

edge_x = []
edge_y = []
edge_z = []
for edge in edges:
    v0, v1 = verts[edge[0]], verts[edge[1]]
    edge_x.extend([v0[0], v1[0], None])
    edge_y.extend([v0[1], v1[1], None])
    edge_z.extend([v0[2], v1[2], None])

fig.add_trace(go.Scatter3d(
    x=edge_x, y=edge_y, z=edge_z,
    mode='lines',
    line=dict(color='darkgray', width=2),
    name='Mesh Edges'
))

# Plot Trajectories
fig.add_trace(go.Scatter3d(
    x=traj_eucl.xs[:,0], y=traj_eucl.xs[:,1], z=traj_eucl.xs[:,2],
    mode='lines', line=dict(color='black', width=5, dash='dash'), name='Discrete Euclidean'
))
fig.add_trace(go.Scatter3d(
    x=traj_rand.xs[:,0], y=traj_rand.xs[:,1], z=traj_rand.xs[:,2],
    mode='lines', line=dict(color='blue', width=6), name='Discrete Randers (Wind Optimized)'
))

# Wind visualization
theta = np.linspace(0, 2*np.pi, 20)
equator_pts = np.stack([np.cos(theta), np.sin(theta), np.zeros_like(theta)], axis=1)
wind_vecs = np.array(jax.vmap(w_net)(jnp.array(equator_pts)))

fig.add_trace(go.Cone(
    x=equator_pts[:,0], y=equator_pts[:,1], z=equator_pts[:,2],
    u=wind_vecs[:,0], v=wind_vecs[:,1], w=wind_vecs[:,2],
    sizemode='absolute', sizeref=0.3, colorscale='Blues', showscale=False, name='Wind'
))

fig.update_layout(title="Discrete Zermelo Navigation on a Mesh", scene=dict(aspectmode='data'))
fig.show()


TypeError: unhashable type: 'jaxlib._jax.ArrayImpl'